In [ ]:
import os
import importlib
import pprint

import matplotlib.pyplot as plt
#%matplotlib widget
%matplotlib inline

import math
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic, binned_statistic_2d
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack, hstack

In [ ]:
from astropy.stats import sigma_clipped_stats

In [ ]:
from astropy.visualization import LinearStretch, LuptonAsinhZscaleStretch, SinhStretch
from astropy.visualization import imshow_norm, simple_norm
from astropy.visualization import MinMaxInterval, ZScaleInterval, PercentileInterval
from astropy.visualization import SqrtStretch, ImageNormalize

In [ ]:
import lsst.afw.image as afwImage
import lsst.afw.display as afwDisplay
import lsst.geom
import lsst.afw.geom as afwGeom

import lsst.daf.butler as dafButler
import lsst.pipe.base

In [ ]:
from lsst.analysis.ap import apdb
from lsst.analysis.ap import nb_utils
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles,
)
from lsst.utils.plotting import stars_cmap
from lsst.utils.plotting import publication_plots
publication_plots.set_rubin_plotstyle()

clrs = get_multiband_plot_colors()
bands_dict = publication_plots.get_band_dicts()

In [ ]:
from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib.ticker import PercentFormatter
import matplotlib.gridspec as gridspec

In [ ]:
import matplotlib.patheffects as pathEffects

from lsst.utils.plotting import (                                                                             
         galaxies_cmap,                                                                                        
         galaxies_color,                                                                                       
         make_figure,                                                                                          
         stars_cmap,                                                                                           
         stars_color,                                                                                          
         set_rubin_plotstyle,                                                                                  
         divergent_cmap,                                                                                       
         accent_color,)    
set_rubin_plotstyle()

In [ ]:
# All figures in this directory
from pathlib import Path

In [ ]:
afwDisplay.setDefaultBackend("firefly")

In [ ]:
butler = dafButler.Butler('dp1', collections='LSSTComCam/DP1')

In [ ]:
from lsst.rsp import get_tap_service
service = get_tap_service("tap")
assert service is not None

lsst_mag_limit = 20.5

query = (
    "SELECT sourceId, visit, ra, dec, psfFlux, psfFluxErr, band, detector " 
    "FROM dp1.Source " 
    "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), CIRCLE('ICRS', 53, -28, 1.75))=1  " 
    f"AND psfFLux > {10 **(-0.4 * (lsst_mag_limit - 31.4))} " 
    "AND (band = 'r' OR band = 'g') "
    "ORDER BY psfFlux ASC "
        )

job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
results = job.fetch_result().to_table()

In [ ]:
butler.registry.queryDatasetTypes('*object*')

In [ ]:
len(set(results['sourceId']))

In [ ]:
len(results)

In [ ]:
# stardata = pd.read_csv('darknet_buyed_table.csv')
stardata = pd.read_csv('table_for_visu_inspect.csv')

In [ ]:
stardata.columns

In [ ]:
plt.hist(stardata['psfFlux'], log=True)

In [ ]:
stardata['psfFluxSNR'] = stardata['psfFlux']/stardata['psfFluxErr']
stardata = stardata[stardata['psfFluxSNR'] > 3.]
stardata = stardata[stardata['band']=='g']

In [ ]:
# stardata['psfFlux_mag'] = (stardata['psfFlux'].values*u.nanojansky).to(u.ABmag).value
mag_error_plus = ((stardata['psfFlux']+stardata['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error_minus = ((stardata['psfFlux']-stardata['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error = 0.5 * (mag_error_minus - mag_error_plus)
stardata['mag_error'] = mag_error

In [ ]:
stardata

In [ ]:
plt.hist(stardata['psfFlux']/stardata['psfFluxErr'])
plt.xlabel("SNR PSF Flux")

In [ ]:
stardata['psfFlux_mag'] = (stardata['psfFlux'].values*u.nanojansky).to(u.ABmag).value
mag_error_plus = ((stardata['psfFlux']+stardata['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error_minus = ((stardata['psfFlux']-stardata['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error = 0.5 * (mag_error_minus - mag_error_plus)
stardata['mag_error'] = mag_error
stardata['gaia_color'] = stardata['gaia_bp_mag'] - stardata['gaia_rp_mag']
stardata['delta_mag'] = stardata['mag'] - stardata['gaia_g_mag']

In [ ]:
len(stardata)

In [ ]:
plt.scatter(stardata['gaia_g_mag'], stardata['delta_mag'], c=stardata['gaia_color'], alpha=0.1, s=3)
plt.xlabel('Gaia g mag')
plt.ylabel('LSST g mag - Gaia g mag')

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
from scipy import linalg

In [ ]:
from scipy.stats import linregress

In [ ]:
stardata

In [ ]:
lsstmags = (stardata.psfFlux.values * u.nanojansky).to(u.ABmag).value

In [ ]:
sigma_clipped_stats(stardata.psfFlux_mag.values - lsstmags)

In [ ]:
from sklearn.linear_model import RANSACRegressor

In [ ]:
# for (vn, dt), agroup in stardata.groupby(['visit', 'detector']):
for vn, agroup in stardata.groupby('visit'):
    df = agroup[agroup['mag']>17]
    df = df[np.abs(df['gaia_color']) < 1.5]
    df = df[(df['psfFlux']/df['psfFluxErr']) > 10]
    if len(df) < 100:
        break
    x = df['gaia_color'].values
    y = df['delta_mag'].values
    # reg = LinearRegression().fit(x.reshape(-1, 1), y.reshape(-1, 1))
    # slp = reg.coef_ 
    # zp = reg.intercept_
    
    reg = RANSACRegressor(estimator=LinearRegression())
    res = reg.fit(x.reshape(-1, 1), y.reshape(-1, 1))
    slp = res.estimator_.coef_[0]
    zp = res.estimator_.intercept_
    # reg = linregress(x, y)
    # slp = reg.slope
    # zp = reg.intercept
    
    lsst_g = agroup['mag'].values - zp - slp * agroup['gaia_color'].values
    stardata.loc[agroup.index, 'g_ransac'] = lsst_g


In [ ]:
print(zp, slp)

In [ ]:
plt.scatter(x, y, c=df['mag'].values, alpha=0.9, s=10)
# plt.plot(np.arange(10, 23), np.arange(10, 23), 'k-')

In [ ]:
plt.scatter(agroup['gaia_g_mag'], lsst_g - agroup['gaia_g_mag'].values, c=agroup['gaia_color'], alpha=0.9, s=10)
# plt.plot(np.arange(10, 23), np.arange(10, 23), 'k-')

In [ ]:
plt.scatter(agroup['gaia_g_mag'], lsst_g, c=agroup['gaia_color'], alpha=0.9, s=10)
plt.plot(np.arange(10, 23), np.arange(10, 23), 'k-')

In [ ]:
plt.scatter(stardata['gaia_g_mag'], stardata['g_ransac'], c=stardata['gaia_color'], alpha=0.9, s=10)
plt.plot(np.arange(10, 23), np.arange(10, 23), 'k-')

In [ ]:
# slope = 0.78349
# zp = -0.1099

In [ ]:
zp

In [ ]:
slp

In [ ]:
g_lsst = stardata['mag'] - zp - slp * stardata['gaia_color']

In [ ]:
plt.scatter(stardata['gaia_g_mag'], g_lsst - stardata['gaia_g_mag'], c=stardata['gaia_color'], alpha=0.2, s=3)
# plt.ylim(-1, 2)
plt.colorbar(label='gaia Color')
plt.xlabel('gaia g mag')
plt.ylabel('lsst g - gaia g')

In [ ]:
plt.scatter(stardata['gaia_g_mag'], stardata['g_ransac'] - stardata['gaia_g_mag'], 
            c=stardata['gaia_color'], alpha=0.1, s=3)
# plt.ylim(-1, 2)
plt.colorbar(label='gaia Color')
plt.xlabel('gaia g mag')
plt.ylabel('lsst g - gaia g')

In [ ]:
# the single visit zp and color term is discrepant with the global zp and color term of zp=-0.3 slp=0.85
# only at the level we see here, around a +/- 0.2 mag, but usually biased towards positive values
# which means that the g_ransac is always getting fainter mags. 
plt.scatter(g_lsst, stardata['g_ransac'] - g_lsst, c=stardata['gaia_color'], alpha=0.1, s=5)
plt.colorbar()

In [ ]:
# the scatter is basically dominated by stars that are red. 
# so there is a color term not being accounted for.
blue_stars = stardata['gaia_color'] < 1.8
plt.scatter(g_lsst[blue_stars], 
            stardata['g_ransac'][blue_stars] - g_lsst[blue_stars], 
            c=stardata['gaia_color'][blue_stars], 
            alpha=0.1, s=5)
plt.colorbar()

In [ ]:
plt.scatter(stardata[blue_stars]['gaia_g_mag'], 
            stardata[blue_stars]['g_ransac'] - stardata[blue_stars]['gaia_g_mag'], 
            c=stardata[blue_stars]['gaia_color'], alpha=0.1, s=3)
# plt.ylim(-1, 2)
plt.colorbar(label='gaia Color')
plt.xlabel('gaia g mag')
plt.ylabel('lsst g - gaia g')

In [ ]:
# selection of very offset stars

offset_stars = stardata[
    (stardata['gaia_g_mag'] < 15) & \
    (np.abs(stardata['g_ransac'] - stardata['gaia_g_mag']) > 0.3)
    ].copy()

In [ ]:
len(offset_stars)

In [ ]:
selection = offset_stars.sort_values('gaia_g_mag', ascending=True)

In [ ]:
from IPython.display import display as ipydisplay

In [ ]:
selection

## Use now the Monster

In [ ]:
butler.registry.queryDatasetTypes('*monster*')

In [ ]:
monsterquery = butler.registry.queryDatasets('the_monster_20250219')
monsterquery.count()
# list(butler.registry.queryDatasets('the_monster_20250219'))

In [ ]:
butler.query_datasets('skyMap')

In [ ]:
skymap = butler.get('skyMap')

In [ ]:
import esutil

In [ ]:
htm = esutil.htm.HTM(7)
htm_indices = set(htm.lookup_id(stardata["ra"], stardata["dec"]))  # type: ignore

In [ ]:
htm_indices

In [ ]:
monster_cats = [butler.get('the_monster_20250219', htm7=htm_indx).asAstropy() for htm_indx in htm_indices]

In [ ]:
monsterCat = vstack(monster_cats)

In [ ]:
len(monsterCat)

In [ ]:
monsterCat['coord_ra_deg'] = np.rad2deg(monsterCat['coord_ra'])
monsterCat['coord_dec_deg'] = np.rad2deg(monsterCat['coord_dec'])

In [ ]:
stars = SkyCoord(ra=stardata['ra'].values*u.deg, dec=stardata['dec'].values*u.deg)

In [ ]:
# Convert to SkyCoord
stars = SkyCoord(ra=stardata['ra'].values*u.deg, dec=stardata['dec'].values*u.deg)
monsters = SkyCoord(ra=monsterCat['coord_ra_deg'].value*u.deg, dec=monsterCat['coord_dec_deg'].value*u.deg)

# Crossmatch (one-to-one, find nearest neighbor)
idx, d2d, d3d = stars.match_to_catalog_sky(monsters)

# Example: Add matched index and separation to stardata
stardata['monsterCat_index'] = idx
stardata['match_sep_arcsec'] = d2d.arcsec
stardata['match_monsterCat'] = d2d.arcsec < 1.

# Merge matched rows for more information (optional)
matched_monsters = monsterCat[idx]
crossmatched = hstack([Table.from_pandas(stardata), matched_monsters])


In [ ]:
crossmatched.columns

In [ ]:
cols = [
    'sourceId','visit','ra','dec','psfFlux','psfFluxErr',
    'band','detector','mag',
    'psfFluxSNR','psfFlux_mag','mag_error','delta_mag','g_ransac',
    'gaia_g_mag','gaia_rp_mag','gaia_bp_mag','gaia_color',
    'monsterCat_index','match_sep_arcsec','match_monsterCat',
    'id','coord_ra','coord_dec',
    'coord_raErr','coord_decErr','epoch',
    'phot_g_mean_flux','phot_bp_mean_flux','phot_rp_mean_flux','phot_g_mean_fluxErr',
    'phot_bp_mean_fluxErr','phot_rp_mean_fluxErr',
    'monster_ComCam_g_flux','monster_ComCam_g_fluxErr','monster_ComCam_g_source_flag',
    'monster_ComCam_i_flux','monster_ComCam_i_fluxErr','monster_ComCam_i_source_flag',
    'monster_ComCam_r_flux','monster_ComCam_r_fluxErr','monster_ComCam_r_source_flag',
    'monster_ComCam_y_flux','monster_ComCam_y_fluxErr','monster_ComCam_y_source_flag',
    'monster_ComCam_z_flux','monster_ComCam_z_fluxErr','monster_ComCam_z_source_flag',
    'monster_DES_g_flux','monster_DES_g_fluxErr',
]

In [ ]:
selection = crossmatched[cols]
selection['monster_ComCam_g_mag'] = (selection['monster_ComCam_g_flux']).to(u.ABmag).value
# selection of very offset stars
# offset_stars = stardata[
#     (stardata['gaia_g_mag'] < 15) & \
#     (np.abs(stardata['g_ransac'] - stardata['gaia_g_mag']) > 0.3)
#     ].copy()

In [ ]:
plt.scatter(selection['monster_ComCam_g_mag'], 
            -2.5*np.log10(selection['psfFlux']/selection['monster_ComCam_g_flux']), 
            c=selection['gaia_color'], alpha=0.2, s=3)
plt.ylim(-0.25, 5)
plt.xlim(12, 21)
plt.colorbar(label='gaia Color')
plt.xlabel('monster g mag')
plt.ylabel('ComCam g - monster g')
# plt.gca().set_xscale('log')
# plt.gca().set_yscale('symlog')
plt.grid()

In [ ]:
selection.columns

In [ ]:
len(selection['visit'])

In [ ]:
len(set(selection['visit']))

In [ ]:
selection['visit'][0]

In [ ]:
avisit = selection[selection['visit']==2024121000409]

In [ ]:
plt.scatter(avisit['monster_ComCam_g_mag'], 
            -2.5*np.log10(avisit['psfFlux']/avisit['monster_ComCam_g_flux']), 
            c=avisit['gaia_color'], alpha=0.2, s=3)
plt.ylim(-0.25, 5)
plt.xlim(12, 21)
plt.colorbar(label='gaia Color')
plt.xlabel('monster g mag')
plt.ylabel('ComCam g - monster g')

In [ ]:
flt = (selection['monster_ComCam_g_mag'] > 15.07)&(selection['monster_ComCam_g_mag'] < 15.08)

In [ ]:
set(selection[flt]['monsterCat_index'])

In [ ]:
astar = selection[selection['monsterCat_index']==5929]

In [ ]:
len(set(astar['visit']))

In [ ]:
astar[['visit', 'detector']]

In [ ]:
plt.scatter(astar['monster_ComCam_g_mag'], 
            -2.5*np.log10(astar['psfFlux']/astar['monster_ComCam_g_flux']), 
            c=astar['gaia_color'], alpha=0.2, s=3)
plt.ylim(-0.25, 5)
plt.xlim(12, 21)
plt.colorbar(label='gaia Color')
plt.xlabel('monster g mag')
plt.ylabel('ComCam g - monster g')

In [ ]:
service = get_tap_service("tap")
assert service is not None

In [ ]:
visit_list = ",".join([str(avisit) for avisit in set(astar['visit']) ] )

In [ ]:
query = (
    "SELECT * FROM dp1.Visit"
    "WHERE dp1.Visit.visit IN ("+visit_list+")"
)

In [ ]:
query

In [ ]:
query = (
    "SELECT * FROM dp1.Visit WHERE dp1.Visit.visit IN (2024120600075,2024120600077,2024120600078,2024120600079,2024120600090,2024120600091,2024120600092,2024120600093,2024120600094,2024120300119,2024120300120,2024120300121,2024120300123,2024120800350,2024120800353,2024120800354,2024120300134,2024120300135,2024120300136,2024120300137,2024120300138,2024120300149,2024112900214,2024120300151,2024112900216,2024112900217,2024112900218,2024120300153,2024120300152,2024120300164,2024112900229,2024120300166,2024120300167,2024112900232,2024112900231,2024120300168,2024112900233,2024120800405,2024120800406,2024112900262,2024112900263,2024112900264,2024112900265,2024112900266,2024120600244,2024112900277,2024112900278,2024120600247,2024112900280,2024112900281,2024112900279,2024120600248,2024120600246,2024112900292,2024120600261,2024112900294,2024120600263,2024120600262,2024112900293,2024120900314,2024120900315,2024120900316,2024120900317,2024120900318,2024113000167,2024113000168,2024113000169,2024113000170,2024113000171,2024120900332,2024120900333,2024120900334,2024112700160,2024112700161,2024112700162,2024112700163,2024112700164,2024112700165,2024112700167,2024112700168,2024112700169,2024112700185,2024112700187,2024112700188,2024120100147,2024120100148,2024120100149,2024120100150,2024120100161,2024120100162,2024120100163,2024120100164,2024120100165,2024120100176,2024120100177,2024120100179,2024120100180,2024120500057,2024120500059,2024120500060,2024120500061,2024120100191,2024120100192,2024120100193,2024120100194,2024120500072,2024120500073,2024120500074,2024120500075,2024120500076,2024112800111,2024112800112,2024112800113,2024112800115,2024112800116,2024112800117,2024112800118,2024112800119,2024120500087,2024120500089,2024120500090,2024120500091,2024120700289,2024112800130,2024120700290,2024120700292,2024120700293,2024112800134,2024112800135,2024120500104,2024120500105,2024112800133,2024120500106,2024112800132,2024120500102,2024110800270,2024110800268,2024120700304,2024120700305,2024110800274,2024120700307,2024120700308,2024110800277,2024120500123,2024110800283,2024120500125,2024110800286,2024120500126,2024120500127,2024120500129,2024120500130,2024110800289,2024110800292,2024120500131,2024120500134,2024110800295,2024120500135,2024120500136,2024120500132,2024120500139,2024120500140,2024120500141,2024120500138,2024120500137,2024110800304,2024120500144,2024120500146,2024110800307,2024120500145,2024120500147,2024110800310,2024120500142,2024112800136,2024110800313,2024112800137,2024110800316,2024112800138,2024121000407,2024121000410,2024121000411,2024121000412,2024121000413,2024121000414,2024121000415,2024121000416,2024121000427,2024121000430,2024121000431,2024121000433,2024121000434) "
)

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
results = job.fetch_result().to_table()

In [ ]:
results_visit = results

In [ ]:
query = (
    "SELECT * FROM dp1.CcdVisit WHERE dp1.CcdVisit.visitId IN (2024120600075,2024120600077,2024120600078,2024120600079,2024120600090,2024120600091,2024120600092,2024120600093,2024120600094,2024120300119,2024120300120,2024120300121,2024120300123,2024120800350,2024120800353,2024120800354,2024120300134,2024120300135,2024120300136,2024120300137,2024120300138,2024120300149,2024112900214,2024120300151,2024112900216,2024112900217,2024112900218,2024120300153,2024120300152,2024120300164,2024112900229,2024120300166,2024120300167,2024112900232,2024112900231,2024120300168,2024112900233,2024120800405,2024120800406,2024112900262,2024112900263,2024112900264,2024112900265,2024112900266,2024120600244,2024112900277,2024112900278,2024120600247,2024112900280,2024112900281,2024112900279,2024120600248,2024120600246,2024112900292,2024120600261,2024112900294,2024120600263,2024120600262,2024112900293,2024120900314,2024120900315,2024120900316,2024120900317,2024120900318,2024113000167,2024113000168,2024113000169,2024113000170,2024113000171,2024120900332,2024120900333,2024120900334,2024112700160,2024112700161,2024112700162,2024112700163,2024112700164,2024112700165,2024112700167,2024112700168,2024112700169,2024112700185,2024112700187,2024112700188,2024120100147,2024120100148,2024120100149,2024120100150,2024120100161,2024120100162,2024120100163,2024120100164,2024120100165,2024120100176,2024120100177,2024120100179,2024120100180,2024120500057,2024120500059,2024120500060,2024120500061,2024120100191,2024120100192,2024120100193,2024120100194,2024120500072,2024120500073,2024120500074,2024120500075,2024120500076,2024112800111,2024112800112,2024112800113,2024112800115,2024112800116,2024112800117,2024112800118,2024112800119,2024120500087,2024120500089,2024120500090,2024120500091,2024120700289,2024112800130,2024120700290,2024120700292,2024120700293,2024112800134,2024112800135,2024120500104,2024120500105,2024112800133,2024120500106,2024112800132,2024120500102,2024110800270,2024110800268,2024120700304,2024120700305,2024110800274,2024120700307,2024120700308,2024110800277,2024120500123,2024110800283,2024120500125,2024110800286,2024120500126,2024120500127,2024120500129,2024120500130,2024110800289,2024110800292,2024120500131,2024120500134,2024110800295,2024120500135,2024120500136,2024120500132,2024120500139,2024120500140,2024120500141,2024120500138,2024120500137,2024110800304,2024120500144,2024120500146,2024110800307,2024120500145,2024120500147,2024110800310,2024120500142,2024112800136,2024110800313,2024112800137,2024110800316,2024112800138,2024121000407,2024121000410,2024121000411,2024121000412,2024121000413,2024121000414,2024121000415,2024121000416,2024121000427,2024121000430,2024121000431,2024121000433,2024121000434) "
)

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
results = job.fetch_result().to_table()

In [ ]:
results_ccd = results

In [ ]:
astar_df = astar.to_pandas()
results_ccd_df = results_ccd.to_pandas()
results_visit_df = results_visit.to_pandas()

In [ ]:
first_df = pd.merge(left=astar_df, right=results_visit_df, on='visit', suffixes=('_star', '_visit'))
merged_df = pd.merge(left=first_df, right=results_ccd_df, left_on=('visit', 'detector'), right_on=('visitId', 'detector'), suffixes=('_firstdf', '_det'))

In [ ]:
merged_df

In [ ]:
merged_df.columns.to_list()

In [ ]:
merged_df['delta_mag'] = -2.5*np.log10(merged_df['psfFlux']/merged_df['monster_ComCam_g_flux'])

In [ ]:
cols_to_corr = ['delta_mag', 'airmass', 'seeing', 'skyNoise', 'zeroPoint', 'skyBg', 'magLim','obsStartMJD_det', 'azimuth', 'psfFluxSNR',]

In [ ]:
corr_matrix = merged_df[merged_df['delta_mag']>0.2][cols_to_corr].corr()

In [ ]:
import seaborn as sns

f, ax = plt.subplots(figsize=(10, 8))
corr = merged_df[merged_df['delta_mag']>0.2][cols_to_corr].corr()
sns.heatmap(corr,
    cmap=sns.diverging_palette(220, 10, as_cmap=True),
    vmin=-1.0, vmax=1.0,
    square=True, ax=ax)

In [ ]:
plt.scatter(merged_df['seeing'],merged_df['delta_mag'], c=merged_df['airmass'], s=10)
plt.xlabel('seeing')
plt.ylabel('delta_mag')
plt.colorbar(label='airmass')
plt.legend()

In [ ]:
plt.scatter(merged_df['airmass'], 
            -2.5*np.log10(merged_df['psfFlux']/merged_df['monster_ComCam_g_flux']), 
            c=merged_df['zeroPoint'], alpha=1, s=10)
# plt.ylim(-0.25, 5)
# plt.xlim(12, 21)
plt.colorbar(label='--')
plt.xlabel('airmass')
plt.ylabel('ComCam g - monster g')

In [ ]:
from scipy.stats import binned_statistic, median_abs_deviation

def binned_running_median_mad(x, y, bins=30, plot=True, ax=None, **plot_kwargs):
    '''
    Compute running median and MAD of y in bins of x using scipy's binned_statistic.
    
    Parameters
    ----------
    x : array-like
        Independent variable.
    y : array-like
        Dependent variable.
    bins : int or sequence
        Number of bins or bin edges.
    plot : bool
        Whether to plot result.
    ax : matplotlib axes, optional
        Axes to plot on.
    **plot_kwargs:
        Style args for ax.plot for median curve.
    
    Returns
    -------
    bin_centers, medians, mads
    '''
    x = np.asarray(x)
    y = np.asarray(y)
    
    # Median in bins
    medians, bin_edges, _ = binned_statistic(x, y, statistic='median', bins=bins)
    bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    
    # MAD in bins
    mads = []
    for i in range(len(bin_edges)-1):
        mask = (x >= bin_edges[i]) & (x < bin_edges[i+1])
        if np.any(mask):
            mads.append(median_abs_deviation(y[mask], scale='normal'))
        else:
            mads.append(np.nan)
    mads = np.array(mads)
    
    if plot:
        if ax is None:
            fig, ax = plt.subplots()
        ax.plot(bin_centers, medians, label='Binned Median', **plot_kwargs)
        ax.fill_between(bin_centers, medians - mads, medians + mads, 
                        color=plot_kwargs.get('color', 'C0'), alpha=0.3, label='Binned MAD')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend()
        # plt.show()
        
    return bin_centers, medians, mads, ax

In [ ]:
xx = selection['monster_ComCam_g_mag'] 
yy = -2.5*np.log10(selection['psfFlux']/selection['monster_ComCam_g_flux'])
cut = np.abs(yy) < 0.75

In [ ]:
# Plot running median & MAD
fig, ax = plt.subplots(1, 1, )
ax.scatter(
    selection['monster_ComCam_g_mag'], 
    -2.5*np.log10(selection['psfFlux']/selection['monster_ComCam_g_flux']), 
    s=2, alpha=0.1, c='gray')

ress = binned_running_median_mad(xx[cut], yy[cut], bins=np.arange(12, 21, 0.3), ax=ax)
ax = ress[-1]
ax.axhline(y=0.3, color='gray', lw=1)
ax.set_xlabel('monster ComCam g mag')
ax.set_ylabel('ComCam g mag - monster ComCam g mag')
ax.set_yscale('symlog')
ax.set_ylim(-0.25, 12)
ax.set_xlim(12, 21)
ax.grid()

In [ ]:
# Plot running median & MAD
xx = selection['monster_ComCam_g_mag'] 
yy = (selection['psfFlux']-selection['monster_ComCam_g_flux'])/selection['psfFluxErr']
cut = np.abs(yy) < 5
fig, ax = plt.subplots(1, 1)
ax.scatter(
    selection['monster_ComCam_g_mag'], 
    (selection['psfFlux']-selection['monster_ComCam_g_flux'])/selection['psfFluxErr'], 
    s=2, alpha=0.1, c='gray')

ress = binned_running_median_mad(xx[cut], yy[cut], bins=np.arange(12, 21, 0.25), ax=ax)
ax = ress[-1]
ax.set_xlabel('monster ComCam g mag')
ax.set_ylabel('Pull g flux')
# ax.set_yscale('symlog')
ax.set_ylim(-15, 15)
ax.set_xlim(12, 21)
ax.grid()

## now use the lsst plot style

In [ ]:
from scipy import stats

In [ ]:
mag_hexbin_xlim_lo = 12
mag_hexbin_xlim_hi = 21
mag_hexbin_ylim_lo = -0.35
mag_hexbin_ylim_hi = 0.7
pulls_hexbin_ylim_lo = -20
pulls_hexbin_ylim_hi = 20
mag_xlim_lo = 12
mag_xlim_hi = 21
mag_ylim_lo = 0.5
mag_ylim_hi = 1

# First the plot on mags
xbins = np.logspace(np.log10(15), np.log10(21), num=40)
ybins = np.arange(-0.8, 1.5, 0.005)

# dimensions of plot: don't touch
fig = plt.figure(figsize=(8, 8))
gs = gridspec.GridSpec(16, 16, hspace=0.01, wspace=0.01)

# Enlarge the figure and grid to accommodate another plot below
fig = plt.figure(figsize=(8, 12))  # double the height
gs = gridspec.GridSpec(24, 16, hspace=0.01, wspace=0.01)  # double the rows

# Define axes for the first plot (top, mag)
ax_main_mag = fig.add_subplot(gs[4:14, 0:12])
ax_yhist_mag = fig.add_subplot(gs[4:14, 12:15], sharey=ax_main_mag)
cax = fig.add_subplot(gs[4:14, 15])  # colorbar axis
ax_xhist = fig.add_subplot(gs[0:4, 0:12], sharex=ax_main_mag)
# ax_xcompl = ax_xhist.twinx()

# Define axes for the second plot (bottom, pull)
ax_main_pull = fig.add_subplot(gs[14:24, 0:12])
ax_yhist_pull = fig.add_subplot(gs[14:24, 12:15], sharey=ax_main_pull)
cax2 = fig.add_subplot(gs[14:24, 15])  # colorbar axis for second plot

xx = selection['monster_ComCam_g_mag'] 
yy = -2.5*np.log10(selection['psfFlux']/selection['monster_ComCam_g_flux'])
cut = np.abs(yy) < 1

x = xx[cut]
values = yy[cut]
pulls = (selection['psfFlux']-selection['monster_ComCam_g_flux'])/selection['psfFluxErr']
pulls = pulls[cut]

# just in case of nans
indices = np.isnan(x) | np.isnan(values)
x = x[~indices]
values = values[~indices]
pulls = pulls[~indices]
true_mags = x

# stats on mag
bns = binned_statistic(
    x,
    values,
    statistic="median",
    bins=xbins,
    range=(mag_xlim_lo+2.5, mag_xlim_hi),
)
xbincenters = (bns.bin_edges[1:]+bns.bin_edges[:-1])/2

means = bns.statistic
ax_main_mag.plot(xbincenters, means, color="k", lw=2, label="Running Median")

bns = binned_statistic(
    x,
    values,
    statistic=stats.median_abs_deviation,
    bins=xbins,
    range=(mag_xlim_lo, mag_xlim_hi),
)
stds = bns.statistic
hb = ax_main_mag.hexbin(
    x,
    values,
    cmap=stars_cmap(single_color=True),
    linewidths=0,
    gridsize=150,
    mincnt=2,
    extent=(
        mag_hexbin_xlim_lo, 
        mag_hexbin_xlim_hi, 
        mag_hexbin_ylim_lo, 
        mag_hexbin_ylim_hi),
    edgecolors=None,
)
ax_main_mag.axhline(0, linestyle=":", color="k", zorder=-1)

ax_main_mag.plot(
    xbincenters,
    means+stds,
    color="k",
    label=r"Median $\pm\sigma_{MAD}$",
    linestyle="--",
)
ax_main_mag.plot(
    xbincenters,
    means-stds,
    color="k",
    linestyle="--",
)

ax_xhist.hist(
    x,
    bins=xbins,
    histtype="step",
    lw=2,
    color=stars_color(),
    label="All",
    orientation="vertical",
)

# Shade region to the right of magnitude 22.5 in both plots
# mag_cut = 22.5
# mask = true_mags > 0
# if mag_cut is not None:
#     mask = true_mags < mag_cut
#     ax_main_mag.axvspan(mag_cut, ax_main_mag.get_xlim()[1], color='gray', alpha=0.2, zorder=-2)
#     ax_main_pull.axvspan(mag_cut, ax_main_pull.get_xlim()[1], color='gray', alpha=0.2, zorder=-2)
ax_yhist_mag.hist(
    values,
    bins=ybins,
    histtype="step",
    lw=2,
    color=accent_color(),
    density=True,
    label='Hosted',
    orientation="horizontal",
)
mv, medv, stdv = sigma_clipped_stats(values, sigma=3, maxiters=5)
text_stds = f"$\mu=${mv:.3f}\n$\sigma=${stdv:.3f}"
ax_yhist_mag.text(0.5, 0.85,
    text_stds,
    color=accent_color(),
    rotation="horizontal",
    transform=ax_yhist_mag.transAxes,
    ha="center",
    va="center",
    fontsize=12)

# Now repeat for pulls
bns_pulls = binned_statistic(
    x,
    pulls,
    statistic="median",
    bins=xbins,
    range=(mag_xlim_lo, mag_xlim_hi),
)
means_pulls = bns_pulls.statistic
bns_pulls = binned_statistic(
    x,
    pulls,
    statistic=stats.median_abs_deviation,
    bins=xbins,
    range=(mag_xlim_lo, mag_xlim_hi),
)
stds_pulls = bns_pulls.statistic

hb_pull = ax_main_pull.hexbin(
    x,
    pulls,
    cmap=stars_cmap(single_color=True),
    #bins='log',
    linewidths=0,
    gridsize=150,
    mincnt=2,
    extent=(
        mag_hexbin_xlim_lo, 
        mag_hexbin_xlim_hi, 
        pulls_hexbin_ylim_lo, 
        pulls_hexbin_ylim_hi),
    edgecolors=None,
)
ax_main_pull.axhline(0, linestyle=":", color="k", zorder=-1)
ax_main_pull.plot(xbincenters, means_pulls, color="k", lw=2, label="Running Median")

ax_main_pull.plot(
    xbincenters,
    means_pulls+stds_pulls,
    color="k",
    label=r"Median $\pm\sigma_{MAD}$",
    linestyle="--",
)
ax_main_pull.plot(
    xbincenters,
    means_pulls-stds_pulls,
    color="k",
    linestyle="--",
)
ybins_pulls = np.arange(
    pulls_hexbin_ylim_lo, 
    pulls_hexbin_ylim_hi, 
    0.5
)

ax_yhist_pull.hist(
    pulls,
    bins=ybins_pulls,
    histtype="step",
    lw=2, linestyle='--',
    color="k",
    density=True,
    # label='Hostless',
    orientation="horizontal",
)
ax_main_pull.axhline(1, lw=0.5, color='k', alpha=0.5)
ax_main_pull.axhline(-1, lw=0.5, color='k', alpha=0.5)

mp, medp, stdp = sigma_clipped_stats(pulls, sigma=3, maxiters=5)
text_stds = f"$\mu=${mp:.3f}\n$\sigma=${stdp:.3f}"
ax_yhist_pull.text(0.5, 0.15,
    text_stds,
    color="k",
    rotation="horizontal",
    transform=ax_yhist_pull.transAxes,
    ha="center",
    va="center",
    fontsize=12)

# Colorbar
cb = fig.colorbar(hb, cax=cax)
label = "Points Per Bin"
text = cax.text(0.5, 0.5, label, color="k",
                rotation="vertical",
                transform=cax.transAxes,
                ha="center",
                va="center",
                fontsize=12)
text.set_path_effects([pathEffects.Stroke(linewidth=3, foreground="w"), pathEffects.Normal()])
ax_main_mag.set_xlim(12, 22)
ax_main_mag.set_ylim(-0.102, 0.1502)

cb_pull = fig.colorbar(hb_pull, cax=cax2)
label = "Points Per Bin"
text_pull = cax2.text(0.5, 0.5, label, color="k",
                rotation="vertical",
                transform=cax2.transAxes,
                ha="center",
                va="center",
                fontsize=12)
text_pull.set_path_effects([pathEffects.Stroke(linewidth=3, foreground="w"), pathEffects.Normal()])
ax_main_pull.set_xlim(12, 22)
ax_main_pull.set_ylim(-12, 12)

# Hide duplicated tick labels
ax_xhist.tick_params(labelbottom=False)

ax_main_mag.legend(loc='lower left', ncols=1)
ax_main_mag.set_ylabel("PSF Mag - True Mag (mag)")
ax_main_pull.set_ylabel("$(f_{PSF} - f_{True})/\sigma_{f_{PSF}}$")
ax_main_pull.set_xlabel("True Mag")
ax_xhist.set_ylabel("Count")
# ax_yhist.set_xlabel("Normalized\nCount", fontsize=12)
# plt.savefig(figures_filepath / "hexbin_psf_magpull.pdf")
plt.show()

In [ ]:
# selection of very offset stars
offset_stars = selection[
    (selection['monster_ComCam_g_mag'] < 15.5) & \
    (-2.5*np.log10(selection['psfFlux']/selection['monster_ComCam_g_flux']) > 0.3)
    ].copy()


In [ ]:
offset_stars['flux_log_offset']=-2.5*np.log10(offset_stars['psfFlux']/offset_stars['monster_ComCam_g_flux'])

In [ ]:
offset_stars = offset_stars.to_pandas()

In [ ]:
offset_stars.sort_values('flux_log_offset',  ascending=True, inplace=True)

In [ ]:
offset_stars.sort_values('monster_ComCam_g_mag',  ascending=False, inplace=True)

In [ ]:
offset_stars

In [ ]:
selection.iloc[11].visit

In [ ]:
# for (vn,det), agroup in selection.groupby(['visit', 'detector']):

def visualize_star_image(irow=0, selection_table=offset_stars):
    vn = selection_table.iloc[irow].visit
    det = selection_table.iloc[irow].detector
    
    agroup = selection_table[(selection_table['visit']==vn)&(selection_table['detector']==det)].copy()
    if len(agroup)==0:
        print("group is empty??")
        return None
    print(len(agroup), vn, det)
    vimage = butler.get('visit_image', dataId=dict(visit=vn, detector=det))
    # rawimage = butler.get('preliminary_visit_image', dataId=dict(exposure=vn, detector=det))
    wcs = vimage.getWcs()
    stars_in_visit = stardata[(stardata['visit']==vn)&(stardata['detector']==det)].copy()
    xcoord, ycoord = wcs.skyToPixelArray(stars_in_visit['ra'].values, stars_in_visit['dec'].values, degrees=True)
    stars_in_visit['x'] = xcoord
    stars_in_visit['y'] = ycoord

    xcoord, ycoord = wcs.skyToPixelArray(agroup['ra'].values, agroup['dec'].values, degrees=True)
    agroup['x'] = xcoord
    agroup['y'] = ycoord
    
    display = afwDisplay.Display(frame=0)
    display.scale("asinh", "zscale")
    display.setMaskTransparency(90)
    display.image(vimage)
    for idxstar, astar in agroup.iterrows():
        display.dot('o', astar['x'], astar['y'], 35)

    # display = afwDisplay.Display(frame=1)
    # display.scale("asinh", "zscale")
    # display.setMaskTransparency(90)
    # display.image(rawimage.maskedImage, wcs=wcs)
    # # for idxstar, astar in agroup.iterrows():
    # #     display.dot('o', astar['x'], astar['y'], 35)
    ipydisplay(agroup[['psfFlux', 'gaia_color', 'x', 'y', 'mag', 'monster_ComCam_g_mag']])
    return display, agroup

In [ ]:
display, agroup = visualize_star_image(irow=0)

In [ ]:
display.scale("asinh", "zscale")

In [ ]:
isource = 0
row = selection.iloc[0]

In [ ]:
vimage = butler.get('visit_image', dataId=dict(visit=row['visit'], detector=row['detector']))

In [ ]:
stars_in_visit = stardata[(stardata['visit']==row['visit'])&(stardata['detector']==row['detector'])].copy()

In [ ]:
wcs = vimage.getWcs()

In [ ]:
xcoord, ycoord = wcs.skyToPixelArray(stars_in_visit['ra'].values, stars_in_visit['dec'].values, degrees=True)

In [ ]:
stars_in_visit['x'] = xcoord
stars_in_visit['y'] = ycoord

In [ ]:
stars_in_visit['psfFlux_mag'] = (stars_in_visit['psfFlux'].values*u.nanojansky).to(u.ABmag).value
mag_error_plus = ((stars_in_visit['psfFlux']+stars_in_visit['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error_minus = ((stars_in_visit['psfFlux']-stars_in_visit['psfFluxErr']).values*u.nanojansky).to(u.ABmag).value
mag_error = 0.5 * (mag_error_minus - mag_error_plus)
stars_in_visit['mag_error'] = mag_error
stars_in_visit['gaia_color'] = stars_in_visit['gaia_bp_mag'] - stars_in_visit['gaia_rp_mag']

In [ ]:
stars_in_visit

In [ ]:
plt.hist(stars_in_visit['mag_error'])

In [ ]:
plt.hist(stars_in_visit['mag'] - stars_in_visit['gaia_g_mag'])

In [ ]:
stars_in_visit['delta_mag'] = stars_in_visit['mag'] - stars_in_visit['gaia_g_mag']
stars_in_visit['delta_mag'].describe()

In [ ]:
stars_in_visit.sort_values('delta_mag')

In [ ]:
plt.errorbar(stars_in_visit['gaia_g_mag'], stars_in_visit['psfFlux_mag'], stars_in_visit['mag_error'])

In [ ]:
plt.scatter(stars_in_visit['gaia_g_mag'], stars_in_visit['delta_mag'], c=stars_in_visit['gaia_color'])

In [ ]:
display = afwDisplay.Display(frame=0)

In [ ]:
display = afwDisplay.Display(frame=0)
display.scale("asinh", "zscale")
display.setMaskTransparency(90)
display.image(vimage)
# for astar in stars_in_visit:
#     display.dot('o', astar['x'], astar['y'], 5)

In [ ]:
display.dot?

In [ ]:
visit=row['visit']
detector=row['detector']

In [ ]:
plt.figure(figsize=(10, 11))
imshow_norm(vimage.image.array, 
            interval=ZScaleInterval(contrast=0.5, krej=4, max_reject=0.1), 
            stretch=LinearStretch(), 
            origin='lower', 
            # interpolation='none'
        )
# plt.colorbar(location='right', shrink=0.7)#, aspect=30)
plt.title(f"{visit}-{detector}")
plt.tight_layout()